In [7]:
# pip install pdfplumber pymupdf
from pathlib import Path
import pdfplumber

try:
    import fitz  # PyMuPDF (optional, for image export)
except ImportError:
    fitz = None

def _rows_to_markdown_table(rows):
    """
    Convert a 2D list of rows to a GitHub-flavored Markdown table.
    Assumes first row is header if present; falls back to generic headers otherwise.
    """
    if not rows:
        return ""

    # Replace None with empty strings to avoid 'None' text in Markdown
    rows = [[("" if c is None else str(c)).strip() for c in r] for r in rows]

    header = rows[0]
    # If header looks empty, synthesize generic headers
    if all(cell == "" for cell in header):
        header = [f"col{i+1}" for i in range(len(header))]
        body = rows
    else:
        body = rows[1:]

    sep = ["---"] * len(header)

    lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(sep) + " |",
    ]
    for r in body:
        # pad short rows
        if len(r) < len(header):
            r = r + [""] * (len(header) - len(r))
        lines.append("| " + " | ".join(r) + " |")

    return "\n".join(lines)

def export_markdown(pdf_path, export_images=False):
    pdf_path = Path(pdf_path).expanduser()
    outdir = pdf_path.with_suffix("")
    outdir.mkdir(exist_ok=True)

    # Optional image export
    img_dir = outdir / "images"
    if export_images:
        if fitz is None:
            raise RuntimeError("PyMuPDF not installed. Run: pip install pymupdf")
        img_dir.mkdir(exist_ok=True)

    md_parts = [f"# Extracted content: {pdf_path.name}\n"]

    # ---- Text + Tables ----
    with pdfplumber.open(pdf_path) as pdf:
        for pnum, page in enumerate(pdf.pages, start=1):
            md_parts.append(f"\n---\n\n## Page {pnum}\n")

            text = page.extract_text() or ""
            if text.strip():
                md_parts.append("### Text\n")
                # Use fenced block to preserve layout-ish formatting
                md_parts.append("```\n" + text + "\n```\n")

            tables = page.extract_tables() or []
            for tnum, table in enumerate(tables, start=1):
                md_parts.append(f"\n### Table {tnum}\n")
                md_parts.append(_rows_to_markdown_table(table) + "\n")

    # ---- Optional: Embedded images via PyMuPDF, then embed in Markdown ----
    if export_images:
        md_parts.append("\n---\n\n## Extracted Images\n")
        with fitz.open(pdf_path) as doc:
            for pnum, page in enumerate(doc, start=1):
                page_img_count = 0
                for img_index, (xref, *_) in enumerate(page.get_images(full=True), start=1):
                    info = doc.extract_image(xref)
                    ext = info.get("ext", "png")
                    img_bytes = info["image"]
                    fname = f"p{pnum:03d}_img{img_index:02d}.{ext}"
                    (img_dir / fname).write_bytes(img_bytes)
                    page_img_count += 1
                    md_parts.append(f"**Page {pnum}, Image {img_index}**\n\n")
                    md_parts.append(f"![p{pnum} image {img_index}](images/{fname})\n\n")
                if page_img_count == 0:
                    # (Optional) note pages without images
                    pass

    md_text = "\n".join(md_parts)
    md_path = outdir / f"{pdf_path.stem}.md"
    md_path.write_text(md_text, encoding="utf-8")

    print(f"Markdown written to: {md_path}")
    if export_images:
        print(f"Images saved under: {img_dir}")

# Example usage:
export_markdown("/Users/melindadeng/Code/Clubs/Alliance for Bioversity/Papers/bo1005-leketa-2019.pdf", export_images=True)


Markdown written to: /Users/melindadeng/Code/Clubs/Alliance for Bioversity/Papers/bo1005-leketa-2019/bo1005-leketa-2019.md
Images saved under: /Users/melindadeng/Code/Clubs/Alliance for Bioversity/Papers/bo1005-leketa-2019/images


In [8]:
# pip install pdfplumber pymupdf
from pathlib import Path
import pdfplumber

try:
    import fitz  # PyMuPDF (optional, for image export)
except ImportError:
    fitz = None

def _rows_to_markdown_table(rows):
    """
    Convert a 2D list of rows to a GitHub-flavored Markdown table.
    Assumes first row is header if present; falls back to generic headers otherwise.
    """
    if not rows:
        return ""

    # Replace None with empty strings to avoid 'None' text in Markdown
    rows = [[("" if c is None else str(c)).strip() for c in r] for r in rows]

    header = rows[0]
    # If header looks empty, synthesize generic headers
    if all(cell == "" for cell in header):
        header = [f"col{i+1}" for i in range(len(header))]
        body = rows
    else:
        body = rows[1:]

    sep = ["---"] * len(header)

    lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(sep) + " |",
    ]
    for r in body:
        # pad short rows
        if len(r) < len(header):
            r = r + [""] * (len(header) - len(r))
        lines.append("| " + " | ".join(r) + " |")

    return "\n".join(lines)

def export_markdown(pdf_path, export_images=False):
    pdf_path = Path(pdf_path).expanduser()
    outdir = pdf_path.with_suffix("")
    outdir.mkdir(exist_ok=True)

    # Optional image export
    img_dir = outdir / "images"
    if export_images:
        if fitz is None:
            raise RuntimeError("PyMuPDF not installed. Run: pip install pymupdf")
        img_dir.mkdir(exist_ok=True)

    md_parts = [f"# Extracted content: {pdf_path.name}\n"]

    # ---- Text + Tables ----
    with pdfplumber.open(pdf_path) as pdf:
        for pnum, page in enumerate(pdf.pages, start=1):
            md_parts.append(f"\n---\n\n## Page {pnum}\n")

            text = page.extract_text() or ""
            if text.strip():
                md_parts.append("### Text\n")
                # Use fenced block to preserve layout-ish formatting
                md_parts.append("```\n" + text + "\n```\n")

            tables = page.extract_tables() or []
            for tnum, table in enumerate(tables, start=1):
                md_parts.append(f"\n### Table {tnum}\n")
                md_parts.append(_rows_to_markdown_table(table) + "\n")

    # ---- Optional: Embedded images via PyMuPDF, then embed in Markdown ----
    if export_images:
        md_parts.append("\n---\n\n## Extracted Images\n")
        with fitz.open(pdf_path) as doc:
            for pnum, page in enumerate(doc, start=1):
                page_img_count = 0
                for img_index, (xref, *_) in enumerate(page.get_images(full=True), start=1):
                    info = doc.extract_image(xref)
                    ext = info.get("ext", "png")
                    img_bytes = info["image"]
                    fname = f"p{pnum:03d}_img{img_index:02d}.{ext}"
                    (img_dir / fname).write_bytes(img_bytes)
                    page_img_count += 1
                    md_parts.append(f"**Page {pnum}, Image {img_index}**\n\n")
                    md_parts.append(f"![p{pnum} image {img_index}](images/{fname})\n\n")
                if page_img_count == 0:
                    # (Optional) note pages without images
                    pass

    md_text = "\n".join(md_parts)
    md_path = outdir / f"{pdf_path.stem}.md"
    md_path.write_text(md_text, encoding="utf-8")

    print(f"Markdown written to: {md_path}")
    if export_images:
        print(f"Images saved under: {img_dir}")

# Example usage:
export_markdown("/Users/melindadeng/Code/Clubs/Alliance for Bioversity/Papers/jo1036-ayele-2022.pdf", export_images=True)


Markdown written to: /Users/melindadeng/Code/Clubs/Alliance for Bioversity/Papers/jo1036-ayele-2022/jo1036-ayele-2022.md
Images saved under: /Users/melindadeng/Code/Clubs/Alliance for Bioversity/Papers/jo1036-ayele-2022/images
